# datasets load aur basic setup kar

In [3]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "Data" / "raw"

sales_df = pd.read_csv(DATA_DIR / "sales.csv")
inventory_df = pd.read_csv(DATA_DIR / "inventory.csv")
product_df = pd.read_csv(DATA_DIR / "product_master.csv")
calendar_df = pd.read_csv(DATA_DIR / "calendar.csv")

sales_df.columns = sales_df.columns.str.strip()
inventory_df.columns = inventory_df.columns.str.strip()
product_df.columns = product_df.columns.str.strip()
calendar_df.columns = calendar_df.columns.str.strip()

sales_df["Date"] = pd.to_datetime(sales_df["Date"])
inventory_df["Snapshot_Date"] = pd.to_datetime(inventory_df["Snapshot_Date"])
product_df["Launch_Date"] = pd.to_datetime(product_df["Launch_Date"])

if "date" in calendar_df.columns:
    calendar_df["date"] = pd.to_datetime(calendar_df["date"])
elif "Date" in calendar_df.columns:
    calendar_df["Date"] = pd.to_datetime(calendar_df["Date"])

print("Project Root:", PROJECT_ROOT)
print("Data Directory:", DATA_DIR)

print("\nDataset Shapes:")
print("Sales:", sales_df.shape)
print("Inventory:", inventory_df.shape)
print("Products:", product_df.shape)
print("Calendar:", calendar_df.shape)

print("\nCalendar Columns:")
print(calendar_df.columns.tolist())

print("\nData loaded successfully.")

Project Root: d:\Zidio project\FORESIGHT Project
Data Directory: d:\Zidio project\FORESIGHT Project\Data\raw

Dataset Shapes:
Sales: (36550, 6)
Inventory: (4800, 8)
Products: (50, 8)
Calendar: (731, 11)

Calendar Columns:
['date', 'year', 'month', 'quarter', 'week', 'day_of_week', 'is_weekend', 'season', 'holiday', 'is_holiday', 'promotion_event']

Data loaded successfully.


# Date Features

In [4]:
sales_df["Year"] = sales_df["Date"].dt.year
sales_df["Month"] = sales_df["Date"].dt.month
sales_df["Quarter"] = sales_df["Date"].dt.quarter
sales_df["Week"] = sales_df["Date"].dt.isocalendar().week.astype(int)
sales_df["Day"] = sales_df["Date"].dt.day
sales_df["Day_of_Week"] = sales_df["Date"].dt.dayofweek
sales_df["Is_Weekend"] = (sales_df["Day_of_Week"] >= 5).astype(int)

sales_df["Month_Sin"] = np.sin(
    2 * np.pi * sales_df["Month"] / 12
)

sales_df["Month_Cos"] = np.cos(
    2 * np.pi * sales_df["Month"] / 12
)

sales_df["Day_of_Week_Sin"] = np.sin(
    2 * np.pi * sales_df["Day_of_Week"] / 7
)

sales_df["Day_of_Week_Cos"] = np.cos(
    2 * np.pi * sales_df["Day_of_Week"] / 7
)

print("Date features created successfully.")

display(
    sales_df[
        [
            "Date",
            "Year",
            "Month",
            "Quarter",
            "Week",
            "Day",
            "Day_of_Week",
            "Is_Weekend",
            "Month_Sin",
            "Month_Cos",
            "Day_of_Week_Sin",
            "Day_of_Week_Cos"
        ]
    ].head()
)

Date features created successfully.


,Date,Year,Month,Quarter,Week,Day,Day_of_Week,Is_Weekend,Month_Sin,Month_Cos,Day_of_Week_Sin,Day_of_Week_Cos
0,2024-01-01,2024,1,1,1,1,0,0,0.5,0.866025,0.0,1.0
1,2024-01-01,2024,1,1,1,1,0,0,0.5,0.866025,0.0,1.0
2,2024-01-01,2024,1,1,1,1,0,0,0.5,0.866025,0.0,1.0
3,2024-01-01,2024,1,1,1,1,0,0,0.5,0.866025,0.0,1.0
4,2024-01-01,2024,1,1,1,1,0,0,0.5,0.866025,0.0,1.0


# calendar features

In [5]:
calendar_features = calendar_df.copy()

calendar_date_col = "date" if "date" in calendar_features.columns else "Date"

calendar_features[calendar_date_col] = pd.to_datetime(
    calendar_features[calendar_date_col]
)

if calendar_date_col != "Date":
    calendar_features = calendar_features.rename(
        columns={calendar_date_col: "Date"}
    )

calendar_features = calendar_features[
    [
        "Date",
        "year",
        "month",
        "quarter",
        "week",
        "day_of_week",
        "is_weekend",
        "season",
        "holiday",
        "is_holiday",
        "promotion_event"
    ]
].drop_duplicates("Date")

sales_df = sales_df.merge(
    calendar_features,
    on="Date",
    how="left",
    suffixes=("", "_calendar")
)

for col in ["holiday", "promotion_event"]:
    if col in sales_df.columns:
        sales_df[col] = sales_df[col].fillna("None")

for col in ["is_holiday"]:
    if col in sales_df.columns:
        sales_df[col] = sales_df[col].fillna(0).astype(int)

print("Calendar features merged successfully.")

print("\nSales Dataset Shape:")
print(sales_df.shape)

print("\nNew Calendar Features:")
display(
    sales_df[
        [
            "Date",
            "season",
            "holiday",
            "is_holiday",
            "promotion_event"
        ]
    ].head(10)
)

Calendar features merged successfully.

Sales Dataset Shape:
(36550, 27)

New Calendar Features:


,Date,season,holiday,is_holiday,promotion_event
0,2024-01-01,Winter,None,0,None
1,2024-01-01,Winter,None,0,None
2,2024-01-01,Winter,None,0,None
3,2024-01-01,Winter,None,0,None
4,2024-01-01,Winter,None,0,None
5,2024-01-01,Winter,None,0,None
6,2024-01-01,Winter,None,0,None
7,2024-01-01,Winter,None,0,None
8,2024-01-01,Winter,None,0,None
9,2024-01-01,Winter,None,0,None


# Lag Features.

In [6]:
sales_df = sales_df.sort_values(
    ["SKU", "Date"]
).reset_index(drop=True)

sales_df["Lag_1"] = (
    sales_df.groupby("SKU")["Units_Sold"]
    .shift(1)
)

sales_df["Lag_7"] = (
    sales_df.groupby("SKU")["Units_Sold"]
    .shift(7)
)

sales_df["Lag_14"] = (
    sales_df.groupby("SKU")["Units_Sold"]
    .shift(14)
)

sales_df["Lag_28"] = (
    sales_df.groupby("SKU")["Units_Sold"]
    .shift(28)
)

print("Lag features created successfully.")

display(
    sales_df[
        [
            "Date",
            "SKU",
            "Units_Sold",
            "Lag_1",
            "Lag_7",
            "Lag_14",
            "Lag_28"
        ]
    ].head(20)
)

Lag features created successfully.


,Date,SKU,Units_Sold,Lag_1,Lag_7,Lag_14,Lag_28
0,2024-01-01,SKU001,5,NaN,NaN,NaN,NaN
1,2024-01-02,SKU001,13,5.0,NaN,NaN,NaN
2,2024-01-03,SKU001,12,13.0,NaN,NaN,NaN
3,2024-01-04,SKU001,23,12.0,NaN,NaN,NaN
4,2024-01-05,SKU001,16,23.0,NaN,NaN,NaN
5,2024-01-06,SKU001,18,16.0,NaN,NaN,NaN
6,2024-01-07,SKU001,19,18.0,NaN,NaN,NaN
7,2024-01-08,SKU001,12,19.0,5.0,NaN,NaN
8,2024-01-09,SKU001,10,12.0,13.0,NaN,NaN
9,2024-01-10,SKU001,11,10.0,12.0,NaN,NaN


| Feature  | Meaning                         |
| -------- | ------------------------------- |
| `Lag_1`  | Previous available sales record |
| `Lag_7`  | 7 records/periods earlier       |
| `Lag_14` | 14 records/periods earlier      |
| `Lag_28` | 28 records/periods earlier      |


# Rolling Demand Features.

In [7]:
sales_df = sales_df.sort_values(
    ["SKU", "Date"]
).reset_index(drop=True)

grouped_demand = sales_df.groupby("SKU")["Units_Sold"]

sales_df["Rolling_Mean_7"] = (
    grouped_demand
    .transform(lambda x: x.rolling(window=7, min_periods=1).mean())
)

sales_df["Rolling_Mean_14"] = (
    grouped_demand
    .transform(lambda x: x.rolling(window=14, min_periods=1).mean())
)

sales_df["Rolling_Mean_28"] = (
    grouped_demand
    .transform(lambda x: x.rolling(window=28, min_periods=1).mean())
)

sales_df["Rolling_Std_7"] = (
    grouped_demand
    .transform(lambda x: x.rolling(window=7, min_periods=2).std())
)

sales_df["Rolling_Std_14"] = (
    grouped_demand
    .transform(lambda x: x.rolling(window=14, min_periods=2).std())
)

sales_df["Rolling_Std_28"] = (
    grouped_demand
    .transform(lambda x: x.rolling(window=28, min_periods=2).std())
)

print("Rolling demand features created successfully.")

display(
    sales_df[
        [
            "Date",
            "SKU",
            "Units_Sold",
            "Rolling_Mean_7",
            "Rolling_Mean_14",
            "Rolling_Mean_28",
            "Rolling_Std_7",
            "Rolling_Std_14",
            "Rolling_Std_28"
        ]
    ].head(20)
)

Rolling demand features created successfully.


,Date,SKU,Units_Sold,Rolling_Mean_7,Rolling_Mean_14,Rolling_Mean_28,Rolling_Std_7,Rolling_Std_14,Rolling_Std_28
0,2024-01-01,SKU001,5,5.000000,5.000000,5.000000,NaN,NaN,NaN
1,2024-01-02,SKU001,13,9.000000,9.000000,9.000000,5.656854,5.656854,5.656854
2,2024-01-03,SKU001,12,10.000000,10.000000,10.000000,4.358899,4.358899,4.358899
3,2024-01-04,SKU001,23,13.250000,13.250000,13.250000,7.410578,7.410578,7.410578
4,2024-01-05,SKU001,16,13.800000,13.800000,13.800000,6.534524,6.534524,6.534524
5,2024-01-06,SKU001,18,14.500000,14.500000,14.500000,6.090977,6.090977,6.090977
6,2024-01-07,SKU001,19,15.142857,15.142857,15.142857,5.814596,5.814596,5.814596
7,2024-01-08,SKU001,12,16.142857,14.750000,14.750000,4.140393,5.496752,5.496752
8,2024-01-09,SKU001,10,15.714286,14.222222,14.222222,4.644505,5.380004,5.380004
9,2024-01-10,SKU001,11,15.571429,13.900000,13.900000,4.790864,5.173651,5.173651


Ye 6 features banenge:
Rolling_Mean_7 → recent 7 periods ki average demand
Rolling_Mean_14 → recent 14 periods ki average demand
Rolling_Mean_28 → recent 28 periods ki average demand
Rolling_Std_7 → 7 periods ki demand volatility
Rolling_Std_14 → 14 periods ki volatility
Rolling_Std_28 → 28 periods ki volatility

Note: min_periods=1/2 rakha hai taaki starting rows unnecessarily NaN na ho jayein.

# Promotion & Pricing Features

In [8]:
sales_df["Promotion_Flag"] = (
    pd.to_numeric(
        sales_df["Promotion"],
        errors="coerce"
    )
    .fillna(0)
    .astype(int)
)

sales_df["Price_Change"] = (
    sales_df
    .groupby("SKU")["Price"]
    .transform(lambda x: x.diff())
)

sales_df["Price_Change_Percent"] = (
    sales_df
    .groupby("SKU")["Price"]
    .transform(lambda x: x.pct_change() * 100)
)

sales_df["Price_Rolling_Mean_7"] = (
    sales_df
    .groupby("SKU")["Price"]
    .transform(
        lambda x: x.rolling(
            window=7,
            min_periods=1
        ).mean()
    )
)

sales_df["Price_vs_Rolling_Mean"] = (
    sales_df["Price"]
    - sales_df["Price_Rolling_Mean_7"]
)

sales_df["Price_Discount_Percent"] = np.where(
    sales_df["Price_Rolling_Mean_7"] > 0,
    (
        (
            sales_df["Price_Rolling_Mean_7"]
            - sales_df["Price"]
        )
        / sales_df["Price_Rolling_Mean_7"]
    ) * 100,
    0
)

sales_df["Promotion_Units"] = (
    sales_df["Units_Sold"]
    * sales_df["Promotion_Flag"]
)

print("Promotion and pricing features created successfully.")

display(
    sales_df[
        [
            "Date",
            "SKU",
            "Units_Sold",
            "Price",
            "Promotion_Flag",
            "Price_Change",
            "Price_Change_Percent",
            "Price_Rolling_Mean_7",
            "Price_vs_Rolling_Mean",
            "Price_Discount_Percent"
        ]
    ].head(20)
)

Promotion and pricing features created successfully.


,Date,SKU,Units_Sold,Price,Promotion_Flag,Price_Change,Price_Change_Percent,Price_Rolling_Mean_7,Price_vs_Rolling_Mean,Price_Discount_Percent
0,2024-01-01,SKU001,5,3664.05,0,NaN,NaN,3664.05,0.0,0.0
1,2024-01-02,SKU001,13,3664.05,0,0.0,0.0,3664.05,0.0,0.0
2,2024-01-03,SKU001,12,3664.05,0,0.0,0.0,3664.05,0.0,0.0
3,2024-01-04,SKU001,23,3664.05,0,0.0,0.0,3664.05,0.0,0.0
4,2024-01-05,SKU001,16,3664.05,0,0.0,0.0,3664.05,0.0,0.0
5,2024-01-06,SKU001,18,3664.05,0,0.0,0.0,3664.05,0.0,0.0
6,2024-01-07,SKU001,19,3664.05,0,0.0,0.0,3664.05,0.0,0.0
7,2024-01-08,SKU001,12,3664.05,0,0.0,0.0,3664.05,0.0,0.0
8,2024-01-09,SKU001,10,3664.05,0,0.0,0.0,3664.05,0.0,0.0
9,2024-01-10,SKU001,11,3664.05,0,0.0,0.0,3664.05,0.0,0.0


# Isme main features hain:

###### Promotion_Flag → promotion active hai ya nahi
###### Price_Change → previous price se difference
###### Price_Change_Percent → percentage price change
###### Price_Rolling_Mean_7 → recent average price
###### Price_vs_Rolling_Mean → current price average se kitna different hai
###### Price_Discount_Percent → approximate discount/price reduction signal

# Product Master Features.

In [9]:
product_features = product_df[
    [
        "SKU",
        "Product_Name",
        "Category",
        "Subcategory",
        "Cost_Price",
        "Selling_Price",
        "Gross_Margin_Per_Unit"
    ]
].drop_duplicates("SKU")

sales_df = sales_df.merge(
    product_features,
    on="SKU",
    how="left"
)

sales_df["Gross_Margin_Percent"] = np.where(
    sales_df["Selling_Price"] > 0,
    (
        sales_df["Gross_Margin_Per_Unit"]
        / sales_df["Selling_Price"]
    ) * 100,
    0
)

sales_df["Estimated_Gross_Margin"] = (
    sales_df["Units_Sold"]
    * sales_df["Gross_Margin_Per_Unit"]
)

print("Product master features merged successfully.")

print("\nSales Dataset Shape:")
print(sales_df.shape)

display(
    sales_df[
        [
            "SKU",
            "Product_Name",
            "Category",
            "Subcategory",
            "Cost_Price",
            "Selling_Price",
            "Gross_Margin_Per_Unit",
            "Gross_Margin_Percent",
            "Estimated_Gross_Margin"
        ]
    ].head(10)
)

Product master features merged successfully.

Sales Dataset Shape:
(36550, 52)


,SKU,Product_Name,Category,Subcategory,Cost_Price,Selling_Price,Gross_Margin_Per_Unit,Gross_Margin_Percent,Estimated_Gross_Margin
0,SKU001,Product 001,Furniture,Chair,1758.45,3664.05,1905.6,52.008024,9528.0
1,SKU001,Product 001,Furniture,Chair,1758.45,3664.05,1905.6,52.008024,24772.8
2,SKU001,Product 001,Furniture,Chair,1758.45,3664.05,1905.6,52.008024,22867.2
3,SKU001,Product 001,Furniture,Chair,1758.45,3664.05,1905.6,52.008024,43828.8
4,SKU001,Product 001,Furniture,Chair,1758.45,3664.05,1905.6,52.008024,30489.6
5,SKU001,Product 001,Furniture,Chair,1758.45,3664.05,1905.6,52.008024,34300.8
6,SKU001,Product 001,Furniture,Chair,1758.45,3664.05,1905.6,52.008024,36206.4
7,SKU001,Product 001,Furniture,Chair,1758.45,3664.05,1905.6,52.008024,22867.2
8,SKU001,Product 001,Furniture,Chair,1758.45,3664.05,1905.6,52.008024,19056.0
9,SKU001,Product 001,Furniture,Chair,1758.45,3664.05,1905.6,52.008024,20961.6


# Inventory Features Merge

In [10]:
inventory_features = inventory_df[
    [
        "Snapshot_Date",
        "SKU",
        "Current_Stock",
        "On_Order",
        "Lead_Time_Days",
        "Safety_Stock",
        "Reorder_Point",
        "Inventory_Value"
    ]
].copy()

inventory_features = (
    inventory_features
    .sort_values(["SKU", "Snapshot_Date"])
    .drop_duplicates("SKU", keep="last")
)

inventory_features = inventory_features.drop(
    columns=["Snapshot_Date"]
)

sales_df = sales_df.merge(
    inventory_features,
    on="SKU",
    how="left"
)

print("Inventory features merged successfully.")

print("\nSales Dataset Shape:")
print(sales_df.shape)

print("\nInventory Features:")
display(
    sales_df[
        [
            "SKU",
            "Current_Stock",
            "On_Order",
            "Lead_Time_Days",
            "Safety_Stock",
            "Reorder_Point",
            "Inventory_Value"
        ]
    ].head(10)
)

print("\nMissing Inventory Values:")
display(
    sales_df[
        [
            "Current_Stock",
            "On_Order",
            "Lead_Time_Days",
            "Safety_Stock",
            "Reorder_Point",
            "Inventory_Value"
        ]
    ].isna().sum()
)

Inventory features merged successfully.

Sales Dataset Shape:
(36550, 58)

Inventory Features:


,SKU,Current_Stock,On_Order,Lead_Time_Days,Safety_Stock,Reorder_Point,Inventory_Value
0,SKU001,771,147,4,159,289,2815160.01
1,SKU001,771,147,4,159,289,2815160.01
2,SKU001,771,147,4,159,289,2815160.01
3,SKU001,771,147,4,159,289,2815160.01
4,SKU001,771,147,4,159,289,2815160.01
5,SKU001,771,147,4,159,289,2815160.01
6,SKU001,771,147,4,159,289,2815160.01
7,SKU001,771,147,4,159,289,2815160.01
8,SKU001,771,147,4,159,289,2815160.01
9,SKU001,771,147,4,159,289,2815160.01



Missing Inventory Values:


Current_Stock      0
On_Order           0
Lead_Time_Days     0
Safety_Stock       0
Reorder_Point      0
Inventory_Value    0
dtype: int64

# Inventory & Demand Derived Features

In [11]:
sales_df["Average_Daily_Demand"] = (
    sales_df["Rolling_Mean_7"]
    .fillna(sales_df["Rolling_Mean_14"])
    .fillna(sales_df["Rolling_Mean_28"])
    .fillna(sales_df["Units_Sold"])
)

sales_df["Days_of_Stock"] = np.where(
    sales_df["Average_Daily_Demand"] > 0,
    sales_df["Current_Stock"] / sales_df["Average_Daily_Demand"],
    np.nan
)

sales_df["Stock_Gap"] = (
    sales_df["Current_Stock"]
    - sales_df["Safety_Stock"]
)

sales_df["Reorder_Gap"] = (
    sales_df["Current_Stock"]
    - sales_df["Reorder_Point"]
)

sales_df["Total_Available_Stock"] = (
    sales_df["Current_Stock"]
    + sales_df["On_Order"]
)

sales_df["Lead_Time_Demand"] = (
    sales_df["Average_Daily_Demand"]
    * sales_df["Lead_Time_Days"]
)

sales_df["Stock_Cover_After_On_Order"] = np.where(
    sales_df["Average_Daily_Demand"] > 0,
    (
        sales_df["Current_Stock"]
        + sales_df["On_Order"]
    ) / sales_df["Average_Daily_Demand"],
    np.nan
)

sales_df["Safety_Stock_Gap"] = (
    sales_df["Current_Stock"]
    - sales_df["Safety_Stock"]
)

print("Inventory and demand features created successfully.")

display(
    sales_df[
        [
            "Date",
            "SKU",
            "Units_Sold",
            "Average_Daily_Demand",
            "Current_Stock",
            "On_Order",
            "Safety_Stock",
            "Reorder_Point",
            "Days_of_Stock",
            "Stock_Gap",
            "Reorder_Gap",
            "Total_Available_Stock",
            "Lead_Time_Demand",
            "Stock_Cover_After_On_Order"
        ]
    ].head(15)
)

Inventory and demand features created successfully.


,Date,SKU,Units_Sold,Average_Daily_Demand,Current_Stock,On_Order,Safety_Stock,Reorder_Point,Days_of_Stock,Stock_Gap,Reorder_Gap,Total_Available_Stock,Lead_Time_Demand,Stock_Cover_After_On_Order
0,2024-01-01,SKU001,5,5.000000,771,147,159,289,154.200000,612,482,918,20.000000,183.600000
1,2024-01-02,SKU001,13,9.000000,771,147,159,289,85.666667,612,482,918,36.000000,102.000000
2,2024-01-03,SKU001,12,10.000000,771,147,159,289,77.100000,612,482,918,40.000000,91.800000
3,2024-01-04,SKU001,23,13.250000,771,147,159,289,58.188679,612,482,918,53.000000,69.283019
4,2024-01-05,SKU001,16,13.800000,771,147,159,289,55.869565,612,482,918,55.200000,66.521739
5,2024-01-06,SKU001,18,14.500000,771,147,159,289,53.172414,612,482,918,58.000000,63.310345
6,2024-01-07,SKU001,19,15.142857,771,147,159,289,50.915094,612,482,918,60.571429,60.622642
7,2024-01-08,SKU001,12,16.142857,771,147,159,289,47.761062,612,482,918,64.571429,56.867257
8,2024-01-09,SKU001,10,15.714286,771,147,159,289,49.063636,612,482,918,62.857143,58.418182
9,2024-01-10,SKU001,11,15.571429,771,147,159,289,49.513761,612,482,918,62.285714,58.954128


Is Cell 9 se important features
Average_Daily_Demand → recent demand estimate
Days_of_Stock → current stock kitne din chalega
Stock_Gap → safety stock se kitna upar/neeche hai
Reorder_Gap → reorder point se kitna difference hai
Total_Available_Stock → current + incoming stock
Lead_Time_Demand → lead time ke dauran expected demand
Stock_Cover_After_On_Order → incoming stock ke baad expected coverage

# Future Demand Target.

In [12]:
sales_df = sales_df.sort_values(
    ["SKU", "Date"]
).reset_index(drop=True)

sales_df["Target_Demand_7D"] = (
    sales_df
    .groupby("SKU")["Units_Sold"]
    .shift(-7)
)

sales_df["Target_Demand_14D"] = (
    sales_df
    .groupby("SKU")["Units_Sold"]
    .shift(-14)
)

sales_df["Target_Demand_28D"] = (
    sales_df
    .groupby("SKU")["Units_Sold"]
    .shift(-28)
)

print("Future demand targets created successfully.")

display(
    sales_df[
        [
            "Date",
            "SKU",
            "Units_Sold",
            "Target_Demand_7D",
            "Target_Demand_14D",
            "Target_Demand_28D"
        ]
    ].head(20)
)

print("\nTarget Missing Values:")
display(
    sales_df[
        [
            "Target_Demand_7D",
            "Target_Demand_14D",
            "Target_Demand_28D"
        ]
    ].isna().sum()
)

Future demand targets created successfully.


,Date,SKU,Units_Sold,Target_Demand_7D,Target_Demand_14D,Target_Demand_28D
0,2024-01-01,SKU001,5,12.0,15.0,14.0
1,2024-01-02,SKU001,13,10.0,18.0,23.0
2,2024-01-03,SKU001,12,11.0,21.0,7.0
3,2024-01-04,SKU001,23,11.0,17.0,12.0
4,2024-01-05,SKU001,16,14.0,10.0,21.0
5,2024-01-06,SKU001,18,10.0,19.0,24.0
6,2024-01-07,SKU001,19,17.0,17.0,18.0
7,2024-01-08,SKU001,12,15.0,13.0,15.0
8,2024-01-09,SKU001,10,18.0,15.0,18.0
9,2024-01-10,SKU001,11,21.0,14.0,16.0



Target Missing Values:


Target_Demand_7D      350
Target_Demand_14D     700
Target_Demand_28D    1400
dtype: int64

| Column              | Meaning                 |
| ------------------- | ----------------------- |
| `Target_Demand_7D`  | 7 periods ahead demand  |
| `Target_Demand_14D` | 14 periods ahead demand |
| `Target_Demand_28D` | 28 periods ahead demand |


# Final Feature Cleaning & Leakage Check.

In [13]:
target_column = "Target_Demand_7D"

required_columns = [
    "Date",
    "SKU",
    "Units_Sold",
    target_column
]

missing_columns = [
    col for col in required_columns
    if col not in sales_df.columns
]

if missing_columns:
    raise KeyError(
        f"Missing required columns: {missing_columns}"
    )

sales_df = sales_df.sort_values(
    ["SKU", "Date"]
).reset_index(drop=True)

before_rows = len(sales_df)

sales_df = sales_df.dropna(
    subset=[target_column]
).reset_index(drop=True)

after_rows = len(sales_df)

leakage_columns = [
    "Target_Demand_7D",
    "Target_Demand_14D",
    "Target_Demand_28D"
]

feature_excluded_columns = [
    col for col in leakage_columns
    if col in sales_df.columns
]

model_features = [
    col for col in sales_df.columns
    if col not in feature_excluded_columns
]

print("Feature dataset cleaned successfully.")

print(f"\nRows before cleaning : {before_rows:,}")
print(f"Rows after cleaning  : {after_rows:,}")
print(f"Rows removed         : {before_rows - after_rows:,}")

print("\nTarget:")
print(target_column)

print("\nExcluded target columns:")
print(feature_excluded_columns)

print("\nPotential leakage columns excluded from features:")
print(feature_excluded_columns)

print("\nFinal Dataset Shape:")
print(sales_df.shape)

print("\nFinal Feature Columns:")
print(model_features)

Feature dataset cleaned successfully.

Rows before cleaning : 36,550
Rows after cleaning  : 36,200
Rows removed         : 350

Target:
Target_Demand_7D

Excluded target columns:
['Target_Demand_7D', 'Target_Demand_14D', 'Target_Demand_28D']

Potential leakage columns excluded from features:
['Target_Demand_7D', 'Target_Demand_14D', 'Target_Demand_28D']

Final Dataset Shape:
(36200, 69)

Final Feature Columns:
['Date', 'SKU', 'Units_Sold', 'Revenue', 'Price', 'Promotion', 'Year', 'Month', 'Quarter', 'Week', 'Day', 'Day_of_Week', 'Is_Weekend', 'Month_Sin', 'Month_Cos', 'Day_of_Week_Sin', 'Day_of_Week_Cos', 'year', 'month', 'quarter', 'week', 'day_of_week', 'is_weekend', 'season', 'holiday', 'is_holiday', 'promotion_event', 'Lag_1', 'Lag_7', 'Lag_14', 'Lag_28', 'Rolling_Mean_7', 'Rolling_Mean_14', 'Rolling_Mean_28', 'Rolling_Std_7', 'Rolling_Std_14', 'Rolling_Std_28', 'Promotion_Flag', 'Price_Change', 'Price_Change_Percent', 'Price_Rolling_Mean_7', 'Price_vs_Rolling_Mean', 'Price_Discount

# Feature Validation & Readiness Check.

In [14]:
print("=" * 80)
print("FEATURE DATASET VALIDATION")
print("=" * 80)

print("\nDataset Shape:")
print(sales_df.shape)

print("\nDuplicate Rows:")
duplicate_count = sales_df.duplicated().sum()
print(duplicate_count)

print("\nMissing Values:")
missing_summary = (
    sales_df.isna()
    .sum()
    .sort_values(ascending=False)
)

display(
    missing_summary[
        missing_summary > 0
    ].to_frame("Missing_Count")
)

print("\nData Types:")
display(
    sales_df.dtypes.to_frame("Data_Type")
)

print("\nTarget Validation:")

if target_column in sales_df.columns:
    print(f"Target column: {target_column}")
    print(f"Missing targets: {sales_df[target_column].isna().sum():,}")
    print(f"Target minimum: {sales_df[target_column].min():,.2f}")
    print(f"Target maximum: {sales_df[target_column].max():,.2f}")
    print(f"Target mean: {sales_df[target_column].mean():,.2f}")
else:
    raise KeyError(
        f"Target column '{target_column}' not found."
    )

print("\nFeature Count:")
print(f"Total columns: {len(sales_df.columns)}")
print(f"Model features: {len(model_features)}")

print("\nIdentifier Columns:")
print(
    [
        col for col in ["Date", "SKU"]
        if col in sales_df.columns
    ]
)

print("\nNumeric Features:")
numeric_features = sales_df[
    model_features
].select_dtypes(
    include=np.number
).columns.tolist()

print(numeric_features)

print("\nCategorical Features:")
categorical_features = sales_df[
    model_features
].select_dtypes(
    include=["object", "category"]
).columns.tolist()

print(categorical_features)

print("\nFeature Readiness:")

if duplicate_count == 0 and sales_df[target_column].isna().sum() == 0:
    print("✅ Dataset is ready for the next modeling-preparation step.")
else:
    print("⚠️ Dataset requires additional cleaning before modeling.")

print("\nValidation completed.")

FEATURE DATASET VALIDATION

Dataset Shape:
(36200, 69)

Duplicate Rows:
0

Missing Values:


,Missing_Count
Lag_28,1400
Target_Demand_28D,1050
Lag_14,700
Lag_7,350
Target_Demand_14D,350
Price_Change_Percent,50
Price_Change,50
Rolling_Std_7,50
Lag_1,50
Rolling_Std_14,50



Data Types:


,Data_Type
Date,datetime64[us]
SKU,str
Units_Sold,int64
Revenue,float64
Price,float64
...,...
Stock_Cover_After_On_Order,float64
Safety_Stock_Gap,int64
Target_Demand_7D,float64
Target_Demand_14D,float64



Target Validation:
Target column: Target_Demand_7D
Missing targets: 0
Target minimum: 0.00
Target maximum: 56.00
Target mean: 14.01

Feature Count:
Total columns: 69
Model features: 66

Identifier Columns:
['Date', 'SKU']

Numeric Features:
['Units_Sold', 'Revenue', 'Price', 'Promotion', 'Year', 'Month', 'Quarter', 'Week', 'Day', 'Day_of_Week', 'Is_Weekend', 'Month_Sin', 'Month_Cos', 'Day_of_Week_Sin', 'Day_of_Week_Cos', 'year', 'month', 'week', 'is_weekend', 'is_holiday', 'Lag_1', 'Lag_7', 'Lag_14', 'Lag_28', 'Rolling_Mean_7', 'Rolling_Mean_14', 'Rolling_Mean_28', 'Rolling_Std_7', 'Rolling_Std_14', 'Rolling_Std_28', 'Promotion_Flag', 'Price_Change', 'Price_Change_Percent', 'Price_Rolling_Mean_7', 'Price_vs_Rolling_Mean', 'Price_Discount_Percent', 'Promotion_Units', 'Cost_Price', 'Selling_Price', 'Gross_Margin_Per_Unit', 'Gross_Margin_Percent', 'Estimated_Gross_Margin', 'Current_Stock', 'On_Order', 'Lead_Time_Days', 'Safety_Stock', 'Reorder_Point', 'Inventory_Value', 'Average_Daily_De

C:\Users\Harsh darji\AppData\Local\Temp\ipykernel_13308\838301748.py:67: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  ].select_dtypes(


Is Cell 12 kya check karega?
✅ Dataset shape
✅ Duplicate rows
✅ Missing values
✅ Data types
✅ Target validation
✅ Numeric features
✅ Categorical features
✅ Model feature count
✅ Basic modeling readiness

# Model ready feature

In [15]:
target_column = "Target_Demand_7D"

identifier_columns = [
    "Date",
    "SKU"
]

target_columns = [
    "Target_Demand_7D",
    "Target_Demand_14D",
    "Target_Demand_28D"
]

excluded_columns = set(
    identifier_columns + target_columns
)

feature_columns = [
    col for col in sales_df.columns
    if col not in excluded_columns
]

X_raw = sales_df[feature_columns].copy()
y = sales_df[target_column].copy()

categorical_columns = X_raw.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numeric_columns = X_raw.select_dtypes(
    include=[np.number]
).columns.tolist()

X = pd.get_dummies(
    X_raw,
    columns=categorical_columns,
    drop_first=False,
    dtype=int
)

X = X.replace(
    [np.inf, -np.inf],
    np.nan
)

numeric_X_columns = X.select_dtypes(
    include=[np.number]
).columns

X[numeric_X_columns] = X[numeric_X_columns].fillna(
    X[numeric_X_columns].median()
)

print("=" * 80)
print("FINAL MODEL-READY FEATURE MATRIX")
print("=" * 80)

print("\nOriginal Dataset Shape:")
print(sales_df.shape)

print("\nFeature Matrix Shape:")
print(X.shape)

print("\nTarget Shape:")
print(y.shape)

print("\nCategorical Columns Encoded:")
print(categorical_columns)

print("\nNumeric Columns:")
print(len(numeric_columns))

print("\nFinal Feature Count:")
print(X.shape[1])

print("\nRemaining Missing Values:")
print(X.isna().sum().sum())

print("\nTarget Missing Values:")
print(y.isna().sum())

print("\nFeature Matrix Preview:")
display(X.head())

print("\nTarget Preview:")
display(y.head())

print("\nModel-ready dataset prepared successfully.")

C:\Users\Harsh darji\AppData\Local\Temp\ipykernel_13308\1315029734.py:26: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = X_raw.select_dtypes(


FINAL MODEL-READY FEATURE MATRIX

Original Dataset Shape:
(36200, 69)

Feature Matrix Shape:
(36200, 144)

Target Shape:
(36200,)

Categorical Columns Encoded:
['quarter', 'day_of_week', 'season', 'holiday', 'promotion_event', 'Product_Name', 'Category', 'Subcategory']

Numeric Columns:
56

Final Feature Count:
144

Remaining Missing Values:
0

Target Missing Values:
0

Feature Matrix Preview:


,Units_Sold,Revenue,Price,Promotion,Year,Month,Quarter,Week,Day,Day_of_Week,...,Subcategory_Cabinet,Subcategory_Chair,Subcategory_Cookware,Subcategory_Cushion,Subcategory_Lamp,Subcategory_Organizer,Subcategory_Rug,Subcategory_Shelf,Subcategory_Sofa,Subcategory_Table
0,5,18320.25,3664.05,0,2024,1,1,1,1,0,...,0,1,0,0,0,0,0,0,0,0
1,13,47632.65,3664.05,0,2024,1,1,1,2,1,...,0,1,0,0,0,0,0,0,0,0
2,12,43968.60,3664.05,0,2024,1,1,1,3,2,...,0,1,0,0,0,0,0,0,0,0
3,23,84273.15,3664.05,0,2024,1,1,1,4,3,...,0,1,0,0,0,0,0,0,0,0
4,16,58624.80,3664.05,0,2024,1,1,1,5,4,...,0,1,0,0,0,0,0,0,0,0



Target Preview:


0    12.0
1    10.0
2    11.0
3    11.0
4    14.0
Name: Target_Demand_7D, dtype: float64


Model-ready dataset prepared successfully.


# chronological train/test split

In [17]:
sales_df = sales_df.sort_values("Date").reset_index(drop=True)

X = X.loc[sales_df.index].reset_index(drop=True)
y = y.loc[sales_df.index].reset_index(drop=True)

split_index = int(len(sales_df) * 0.80)

train_data = sales_df.iloc[:split_index].copy()
test_data = sales_df.iloc[split_index:].copy()

X_train = X.iloc[:split_index].copy()
X_test = X.iloc[split_index:].copy()

y_train = y.iloc[:split_index].copy()
y_test = y.iloc[split_index:].copy()

print("=" * 80)
print("CHRONOLOGICAL TRAIN / TEST SPLIT")
print("=" * 80)

print("\nTraining Data:")
print(f"Rows  : {len(X_train):,}")
print(f"Start : {train_data['Date'].min()}")
print(f"End   : {train_data['Date'].max()}")

print("\nTesting Data:")
print(f"Rows  : {len(X_test):,}")
print(f"Start : {test_data['Date'].min()}")
print(f"End   : {test_data['Date'].max()}")

print("\nFeature Shapes:")
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

print("\nTarget Shapes:")
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

if train_data["Date"].max() < test_data["Date"].min():
    print("\n✅ Chronological split verified.")
    print("✅ No future dates are included in training data.")
else:
    print("\n⚠️ Date overlap detected.")

print("\nTrain/Test split completed.")

CHRONOLOGICAL TRAIN / TEST SPLIT

Training Data:
Rows  : 28,960
Start : 2024-01-01 00:00:00
End   : 2025-08-02 00:00:00

Testing Data:
Rows  : 7,240
Start : 2025-08-02 00:00:00
End   : 2025-12-24 00:00:00

Feature Shapes:
X_train: (28960, 144)
X_test : (7240, 144)

Target Shapes:
y_train: (28960,)
y_test : (7240,)

⚠️ Date overlap detected.

Train/Test split completed.


# Preprocessing Pipeline.

In [18]:
X_train = X_train.copy()
X_test = X_test.copy()

X_test = X_test.reindex(
    columns=X_train.columns,
    fill_value=0
)

X_train = X_train.replace(
    [np.inf, -np.inf],
    np.nan
)

X_test = X_test.replace(
    [np.inf, -np.inf],
    np.nan
)

train_medians = X_train.median(numeric_only=True)

X_train = X_train.fillna(train_medians)
X_test = X_test.fillna(train_medians)

X_train = X_train.astype(float)
X_test = X_test.astype(float)

print("=" * 80)
print("FEATURE PREPROCESSING")
print("=" * 80)

print("\nX_train Shape:", X_train.shape)
print("X_test Shape :", X_test.shape)

print("\nMissing Values:")
print("X_train:", X_train.isna().sum().sum())
print("X_test :", X_test.isna().sum().sum())

print("\nInfinite Values:")
print("X_train:", np.isinf(X_train).sum().sum())
print("X_test :", np.isinf(X_test).sum().sum())

print("\nFeature Alignment:")
print("Same columns:", X_train.columns.equals(X_test.columns))

print("\nPreprocessing completed successfully.")

FEATURE PREPROCESSING

X_train Shape: (28960, 144)
X_test Shape : (7240, 144)

Missing Values:
X_train: 0
X_test : 0

Infinite Values:
X_train: 0
X_test : 0

Feature Alignment:
Same columns: True

Preprocessing completed successfully.


# Feature Selection

In [19]:
feature_variance = X_train.var().sort_values()

low_variance_features = feature_variance[
    feature_variance <= 0
].index.tolist()

print("=" * 80)
print("FEATURE SELECTION ANALYSIS")
print("=" * 80)

print("\nTotal Features:")
print(len(X_train.columns))

print("\nConstant Features:")
print(len(low_variance_features))

if low_variance_features:
    print("\nConstant Features Found:")
    print(low_variance_features)
else:
    print("\n✅ No constant features found.")

feature_summary = pd.DataFrame({
    "Feature": X_train.columns,
    "Variance": X_train.var().values,
    "Missing_Train": X_train.isna().sum().values,
    "Missing_Test": X_test.isna().sum().values
})

display(
    feature_summary
    .sort_values("Variance")
    .head(20)
)

print("\nFeature selection analysis completed.")

FEATURE SELECTION ANALYSIS

Total Features:
144

Constant Features:
14

Constant Features Found:
['Price_Change', 'Price_Discount_Percent', 'Price_vs_Rolling_Mean', 'Price_Change_Percent', 'Product_Name_Product 046', 'Product_Name_Product 047', 'Product_Name_Product 048', 'Product_Name_Product 042', 'Product_Name_Product 041', 'Product_Name_Product 043', 'Product_Name_Product 044', 'Product_Name_Product 045', 'Product_Name_Product 049', 'Product_Name_Product 050']


,Feature,Variance,Missing_Train,Missing_Test
31,Price_Change,0.000000,0,0
35,Price_Discount_Percent,0.000000,0,0
34,Price_vs_Rolling_Mean,0.000000,0,0
32,Price_Change_Percent,0.000000,0,0
124,Product_Name_Product 046,0.000000,0,0
125,Product_Name_Product 047,0.000000,0,0
126,Product_Name_Product 048,0.000000,0,0
120,Product_Name_Product 042,0.000000,0,0
119,Product_Name_Product 041,0.000000,0,0
121,Product_Name_Product 043,0.000000,0,0



Feature selection analysis completed.


# Remove Constant Features

In [20]:
constant_features = X_train.columns[
    X_train.nunique() <= 1
].tolist()

X_train = X_train.drop(
    columns=constant_features
)

X_test = X_test.drop(
    columns=constant_features
)

model_features = X_train.columns.tolist()

print("=" * 80)
print("FINAL FEATURE SET")
print("=" * 80)

print(f"\nRemoved Constant Features: {len(constant_features)}")

print("\nRemaining Features:")
print(len(model_features))

print("\nX_train Shape:")
print(X_train.shape)

print("\nX_test Shape:")
print(X_test.shape)

print("\nFeature Alignment:")
print(X_train.columns.equals(X_test.columns))

print("\nRemaining Missing Values:")
print("X_train:", X_train.isna().sum().sum())
print("X_test :", X_test.isna().sum().sum())

print("\nFeature set finalized successfully.")

FINAL FEATURE SET

Removed Constant Features: 14

Remaining Features:
130

X_train Shape:
(28960, 130)

X_test Shape:
(7240, 130)

Feature Alignment:
True

Remaining Missing Values:
X_train: 0
X_test : 0

Feature set finalized successfully.


# Save Final Engineered Dataset

In [21]:
OUTPUT_DIR = PROJECT_ROOT / "models" / "features"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

X_train_path = OUTPUT_DIR / "X_train.csv"
X_test_path = OUTPUT_DIR / "X_test.csv"
y_train_path = OUTPUT_DIR / "y_train.csv"
y_test_path = OUTPUT_DIR / "y_test.csv"
feature_list_path = OUTPUT_DIR / "feature_columns.csv"

X_train.to_csv(X_train_path, index=False)
X_test.to_csv(X_test_path, index=False)

y_train.to_csv(y_train_path, index=False, header=True)
y_test.to_csv(y_test_path, index=False, header=True)

pd.DataFrame({
    "Feature": model_features
}).to_csv(
    feature_list_path,
    index=False
)

print("=" * 80)
print("FEATURE DATASETS SAVED")
print("=" * 80)

print("\nSaved files:")

for path in [
    X_train_path,
    X_test_path,
    y_train_path,
    y_test_path,
    feature_list_path
]:
    print(path)

print("\nFeature count:", len(model_features))
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

print("\n✅ Feature engineering pipeline completed successfully.")

FEATURE DATASETS SAVED

Saved files:
d:\Zidio project\FORESIGHT Project\models\features\X_train.csv
d:\Zidio project\FORESIGHT Project\models\features\X_test.csv
d:\Zidio project\FORESIGHT Project\models\features\y_train.csv
d:\Zidio project\FORESIGHT Project\models\features\y_test.csv
d:\Zidio project\FORESIGHT Project\models\features\feature_columns.csv

Feature count: 130
Training rows: 28960
Testing rows: 7240

✅ Feature engineering pipeline completed successfully.
